In [ ]:
#!/usr/bin/env python3
"""
Downsample DeepLabCut keypoint-tracking CSV files to a common frame rate.

The recording FPS is NOT stored inside the DLC .csv or .h5 files, but DeepLabCut
DOES write it into the companion metadata pickle produced by `analyze_videos`
(the "..._meta.pickle", and it is also embedded inside "..._full.pickle").
This script reads the FPS from that pickle automatically for each CSV, then
writes a *downsampled copy* into an output folder. It NEVER modifies originals.

Downsampling: for a 50 FPS video going to 10 FPS we keep every 5th frame
(step = 50 / 10 = 5). For the 27 FPS example, 27/10 -> nearest step 3
(effective 9 FPS); the script warns whenever the ratio isn't exact.

Assumes SINGLE-animal DLC files (3 header rows: scorer / bodyparts / coords).
Header rows are detected automatically.

--------------------------------------------------------------------------------
HOW TO USE
--------------------------------------------------------------------------------
1. Edit the CONFIG section below (OUTPUT_DIR, TARGET_FPS, INPUTS).
2. Run inside your DeepLabCut python environment (it can already unpickle DLC
   files):   python downsample_dlc.py
--------------------------------------------------------------------------------
"""

import csv
import json
import pickle
import sys
from pathlib import Path

# ============================================================================
# CONFIG  --  edit this section, then run the script.
# ============================================================================

# Where the downsampled copies are written. Originals are never touched.
OUTPUT_DIR = r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\unifiedDataset_topview_DLC_singleAnimal\analyzedVideos_unifiedDataset_DLC_singleAnimal_downsampled_10fps"

# The frame rate you want everything downsampled to.
TARGET_FPS = 10

# One entry per input folder to scan for DLC CSVs.
#   folder     : path to search for *.csv
#   recursive  : True searches sub-folders too
#   fps        : OPTIONAL manual override. Leave it out (or None) to auto-read
#                the FPS from each file's companion DLC pickle. Set a number
#                only if the pickle is missing or you want to force a value.
INPUTS = [
    {"folder": r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\unifiedDataset_topview_DLC_singleAnimal\analyzedVideos_trainingSessions_DLC_singleAnimal", "recursive": True},
    {"folder": r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\unifiedDataset_topview_DLC_singleAnimal\analyzedVideos_maladapivePreScreening_DLC_singleAnimal", "recursive": True},
    {"folder": r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\unifiedDataset_topview_DLC_singleAnimal\analyzedVideos_Stefanos_RIdataset_DLC_singleAnimal", "recursive": True},
]

# Which DLC tracking files to downsample:
#   (".h5",)          -> raw DLC predictions  (RECOMMENDED for KP-MoSeq)
#   (".csv",)         -> DLC CSV outputs
#   (".h5", ".csv")   -> both
# Using .h5 gives KP-MoSeq the raw keypoints; it does its own noise modelling
# (calibration + outlier_removal), so raw is preferred over pre-filtered data.
INPUT_EXTS = (".h5",)

# Process the DLC "_filtered" outputs too (True) or skip them (False).
# The raw .h5 has no "_filtered" variant; this mainly affects .csv.
INCLUDE_FILTERED = False

# If a file's FPS cannot be auto-read and no manual override is given:
#   "skip"  -> skip the file with a warning (safe default)
#   "error" -> stop the whole run
ON_MISSING_FPS = "skip"

# What to do when a dataset's FPS is NOT an integer multiple of TARGET_FPS,
# so exact uniform decimation is impossible (e.g. 15 or 27 -> 10):
#   "interpolate" -> time-resample to EXACTLY TARGET_FPS (uniform rate; keeps
#                    all datasets at one frame rate for KP-MoSeq). Coordinates
#                    are linearly interpolated; each resampled confidence is the
#                    MIN of its two bracketing frames (conservative, so points
#                    interpolated across an occlusion are treated as unreliable).
#   "nearest"     -> decimate by the nearest integer step (27->9, 15->7.5 FPS)
#   "skip"        -> skip these files
# Near-integer-multiple datasets are always DECIMATED (see DECIMATE_TOL).
NONMULTIPLE_METHOD = "interpolate"

# A recording is DECIMATED (not interpolated) when some integer step k lands
# the result within this fractional tolerance of TARGET_FPS, i.e.
# |fps/k - TARGET_FPS| <= DECIMATE_TOL * TARGET_FPS. This absorbs measurement
# jitter: 49.48, 49.99, 50.01 FPS all decimate by 5 (~9.9-10.0 FPS) instead of
# being interpolated. 0.05 = accept a decimated rate within 5% of TARGET_FPS.
# With 0.05: ~50 and ~30 decimate; 15 and 27 interpolate; 10.57 (5.7% off 10,
# no integer step reaches 10) is interpolated to exactly 10 -- raise this to
# ~0.06 if you'd rather leave those ~10.6 FPS videos decimated/unchanged.
DECIMATE_TOL = 0.05

# ============================================================================
# END CONFIG  --  you shouldn't need to edit below here.
# ============================================================================


def first_field_is_frame_index(field: str) -> bool:
    """A data row in a DLC CSV starts with an integer frame index."""
    try:
        int(field)
        return True
    except (ValueError, TypeError):
        return False


def _search_pickle_for_fps(obj, _depth=0):
    """Recursively look for an 'fps' entry inside an unpickled DLC object."""
    if _depth > 8:
        return None
    if isinstance(obj, dict):
        for key, val in obj.items():
            if isinstance(key, str) and key.lower() == "fps":
                try:
                    fps = float(val)
                    if fps > 0:
                        return fps
                except (TypeError, ValueError):
                    pass
        for val in obj.values():
            found = _search_pickle_for_fps(val, _depth + 1)
            if found is not None:
                return found
    elif isinstance(obj, (list, tuple)):
        for val in obj:
            found = _search_pickle_for_fps(val, _depth + 1)
            if found is not None:
                return found
    return None


def _pickle_base(path: Path) -> str:
    """Video+scorer prefix shared by a tracking file and its DLC pickle.

    Strips the tracking-file suffixes so an h5 like '<prefix>_sk.h5' or
    '<prefix>_sk_tr.h5' (and filtered '<prefix>_filtered.csv') maps to the
    pickle prefix '<prefix>' used by '<prefix>_meta.pickle'.
    """
    stem = path.stem
    for suf in ("_filtered", "_sk_tr", "_sk"):
        if stem.endswith(suf):
            stem = stem[: -len(suf)]
    return stem


def find_fps_from_pickle(csv_path: Path, input_root: Path):
    """
    Locate the DLC metadata pickle for a tracking file and return its fps.

    Tries '<base>_meta.pickle' / '<base>_full.pickle' next to the file, then a
    prefix match against any *_meta/_full pickle in the same folder (handles
    snapshot/scorer tokens that differ between the h5 and pickle names), then a
    tree-wide search. Returns (fps, pickle_name) or (None, None).
    """
    base = _pickle_base(csv_path)
    candidate_names = [f"{base}_meta.pickle", f"{base}_full.pickle"]

    candidates = []
    for name in candidate_names:
        p = csv_path.parent / name
        if p.is_file():
            candidates.append(p)

    # Same-folder prefix match (robust to differing snapshot/scorer tokens).
    if not candidates:
        for pkl in sorted(list(csv_path.parent.glob("*_meta.pickle"))
                          + list(csv_path.parent.glob("*_full.pickle"))):
            pbase = pkl.stem
            for suf in ("_meta", "_full"):
                if pbase.endswith(suf):
                    pbase = pbase[: -len(suf)]
            if base.startswith(pbase) or pbase.startswith(base):
                candidates.append(pkl)

    # Fall back to a tree-wide search.
    if not candidates:
        for name in candidate_names:
            candidates.extend(input_root.rglob(name))

    for pkl in candidates:
        try:
            with pkl.open("rb") as f:
                data = pickle.load(f)
        except Exception as exc:  # noqa: BLE001
            print(f"    [WARN] could not read {pkl.name}: {exc}")
            continue
        fps = _search_pickle_for_fps(data)
        if fps is not None:
            return fps, pkl.name
    return None, None


def split_header(rows):
    """Return (header_rows, data_rows) splitting on the first frame-index row."""
    n_header = 0
    for row in rows:
        first = row[0] if row else ""
        if first_field_is_frame_index(first):
            break
        n_header += 1
    return rows[:n_header], rows[n_header:]


def resample_keypoints(coords, confs, src_fps, dst_fps):
    """
    Time-resample keypoints from src_fps to EXACTLY dst_fps.

    coords : (M, K, 2) float array of x,y per keypoint per frame
    confs  : (M, K)    float array of likelihoods
    Returns (new_coords (N,K,2), new_confs (N,K), nearest_frames (N,)):
      - coordinates are linearly interpolated in time
      - each resampled confidence is the MIN of its two bracketing source
        frames (conservative: interpolating across an occluded/low-conf frame
        yields a low confidence, so KP-MoSeq down-weights it)
      - nearest_frames[j] is the closest ORIGINAL frame to output frame j,
        used as an (approximate) video_frame_indexes entry.
    """
    import numpy as np

    M = coords.shape[0]
    if M < 2:
        return coords, confs, np.arange(M)
    N = int(np.floor((M - 1) * (dst_fps / src_fps))) + 1
    pos = (np.arange(N) / dst_fps) * src_fps      # fractional source index
    i0 = np.floor(pos + 1e-9).astype(int)
    frac = pos - i0
    i0 = np.clip(i0, 0, M - 1)
    i1 = np.clip(i0 + 1, 0, M - 1)
    exact = frac < 1e-9

    w = frac[:, None, None]
    new_coords = coords[i0] * (1.0 - w) + coords[i1] * w
    new_coords = np.where(exact[:, None, None], coords[i0], new_coords)

    new_confs = np.minimum(confs[i0], confs[i1])
    new_confs = np.where(exact[:, None], confs[i0], new_confs)

    nearest = np.clip(np.rint(pos).astype(int), 0, M - 1)
    return new_coords, new_confs, nearest


def process_csv(src: Path, dst: Path, plan: dict, fps: float):
    """
    Write a downsampled copy of a single-animal DLC .csv to dst.

    - method 'decimate'/'asis': header rows preserved, every step-th data row.
    - method 'interpolate': coordinates time-resampled to exactly TARGET_FPS.

    Returns a dict: kept/total/frames/interpolated.
    """
    with src.open("r", newline="", encoding="utf-8") as fin:
        rows = list(csv.reader(fin))

    header_rows, data_rows = split_header(rows)
    total = len(data_rows)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if plan["method"] == "interpolate":
        import numpy as np

        K = (len(data_rows[0]) - 1) // 3

        def _f(x):
            try:
                return float(x)
            except (ValueError, TypeError):
                return np.nan

        a = np.array([[_f(v) for v in r[1:1 + 3 * K]] for r in data_rows])
        a = a.reshape(total, K, 3)
        nc, ncf, nearest = resample_keypoints(a[:, :, :2], a[:, :, 2], fps, TARGET_FPS)
        out = np.concatenate([nc, ncf[:, :, None]], axis=2).reshape(len(nearest), 3 * K)
        with dst.open("w", newline="", encoding="utf-8") as fout:
            writer = csv.writer(fout)
            writer.writerows(header_rows)
            for j in range(len(nearest)):
                writer.writerow([j] + [repr(float(v)) for v in out[j]])
        return {"kept": len(nearest), "total": total,
                "frames": nearest.tolist(), "interpolated": True}

    # decimate / asis
    step = plan["step"]
    kept = data_rows[::step]
    with dst.open("w", newline="", encoding="utf-8") as fout:
        writer = csv.writer(fout)
        writer.writerows(header_rows)
        writer.writerows(kept)
    return {"kept": len(kept), "total": total,
            "frames": list(range(0, total, step)), "interpolated": False}


def collect_files(folder: Path, recursive: bool) -> list[Path]:
    globber = folder.rglob if recursive else folder.glob
    result = []
    for ext in INPUT_EXTS:
        for f in globber(f"*{ext}"):
            if not f.is_file():
                continue
            if not INCLUDE_FILTERED and f.stem.endswith("_filtered"):
                continue
            result.append(f)
    return sorted(set(result))


def process_h5(src: Path, dst: Path, plan: dict, fps: float):
    """
    Write a downsampled copy of a single-animal DLC .h5 file.

    - method 'decimate'/'asis': keep every step-th frame (positional).
    - method 'interpolate': coordinates time-resampled to exactly TARGET_FPS.
    Returns the same dict shape as process_csv().
    """
    import numpy as np
    import pandas as pd  # lazy: only needed when processing .h5

    df = pd.read_hdf(src)
    total = len(df)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if plan["method"] == "interpolate":
        K = df.shape[1] // 3
        a = df.to_numpy().astype(float).reshape(total, K, 3)
        nc, ncf, nearest = resample_keypoints(a[:, :, :2], a[:, :, 2], fps, TARGET_FPS)
        out = np.concatenate([nc, ncf[:, :, None]], axis=2).reshape(len(nearest), 3 * K)
        df_out = pd.DataFrame(out, columns=df.columns)
        frames = nearest.tolist()
        interpolated = True
    else:
        step = plan["step"]
        df_out = df.iloc[::step]
        frames = list(range(0, total, step))
        interpolated = False

    # DLC's conventional key + table format keeps it readable by DLC & KP-MoSeq.
    df_out.to_hdf(dst, key="df_with_missing", mode="w", format="table")

    return {"kept": len(df_out), "total": total,
            "frames": frames, "interpolated": interpolated}


def plan_for_fps(fps: float) -> dict:
    """Decide how to reach TARGET_FPS from a source fps."""
    ratio = fps / TARGET_FPS
    k = round(ratio)

    # Decimate if an integer step lands within tolerance of TARGET_FPS
    # (absorbs jitter like 49.48/49.99/50.01 -> every 5).
    if k >= 1 and abs(fps / k - TARGET_FPS) <= DECIMATE_TOL * TARGET_FPS:
        eff = fps / k
        note = (f"decimate every {k} (~{eff:.2f} FPS)" if k > 1
                else f"keep all (~{eff:.2f} FPS)")
        return {"method": "decimate", "step": k, "note": note}

    if ratio < 1 - 1e-9:
        return {"method": "asis", "step": 1,
                "note": f"{fps:g} FPS below target; copied unchanged"}
    if NONMULTIPLE_METHOD == "interpolate":
        return {"method": "interpolate", "step": None,
                "note": f"interpolate {fps:g}->{TARGET_FPS} FPS (non-integer ratio)"}
    if NONMULTIPLE_METHOD == "nearest":
        step = max(1, k)
        return {"method": "decimate", "step": step,
                "note": f"nearest decimate every {step} (~{fps/step:.2f} FPS)"}
    return {"method": "skip", "step": None,
            "note": f"{fps:g} FPS not near a multiple of {TARGET_FPS}"}


def main() -> None:
    out_root = Path(OUTPUT_DIR)
    out_root.mkdir(parents=True, exist_ok=True)

    total_files = 0
    skipped = 0
    # recording-name -> list of ORIGINAL video frame numbers per output row.
    # Feed this to KP-MoSeq as `video_frame_indexes` so calibration / grid
    # movies pull the correct full-rate frame WITHOUT re-encoding the videos.
    frame_index_map = {}
    interpolated_recordings = []  # nearest-frame mapping is approximate here
    meta_records = []  # per-session: dataset, recording, source_fps, method, ...
    for entry in INPUTS:
        folder = Path(entry["folder"])
        recursive = entry.get("recursive", True)
        override_fps = entry.get("fps", None)

        if not folder.exists():
            print(f"[WARN] Input folder does not exist, skipping: {folder}")
            continue

        files = collect_files(folder, recursive)
        if not files:
            print(f"[INFO] No CSVs found in {folder}")
            continue

        print(f"\n=== {folder} -> {TARGET_FPS} FPS ===")
        for src in files:
            rel = src.relative_to(folder)

            if override_fps is not None:
                fps, source = float(override_fps), "manual override"
            else:
                fps, pkl_name = find_fps_from_pickle(src, folder)
                source = f"from {pkl_name}" if fps is not None else None

            if fps is None:
                msg = (f"  [SKIP] {rel}: FPS not found in any companion pickle. "
                       f"Set a manual 'fps' for this folder to force a value.")
                if ON_MISSING_FPS == "error":
                    print(msg)
                    sys.exit(1)
                print(msg)
                skipped += 1
                continue

            plan = plan_for_fps(fps)
            if plan["method"] == "skip":
                print(f"  [SKIP] {rel}: {plan['note']}.")
                skipped += 1
                continue

            # Flatten: all mouseID folders live directly under OUTPUT_DIR
            # (rel already starts with the mouseID). Datasets are merged.
            dst = out_root / rel
            if dst.exists():
                print(f"    [WARN] {rel} already exists (same mouse/file in "
                      f"another dataset); overwriting.")
            if src.suffix.lower() == ".h5":
                result = process_h5(src, dst, plan, fps)
            else:
                result = process_csv(src, dst, plan, fps)

            total_files += 1
            print(f"  {rel}: {fps:g} FPS ({source}), {plan['note']} -> "
                  f"{result['kept']}/{result['total']} rows")

            meta_records.append({
                "dataset": folder.name,
                "recording": src.stem,
                "source_fps": fps,
                "target_fps": TARGET_FPS,
                "method": plan["method"],
                "orig_frames": result["total"],
                "kept_frames": result["kept"],
                "fps_source": source,
                "relpath": str(rel),
            })

            # KP-MoSeq names each recording by the filename (minus extension),
            # scorer suffix included. Key the map the same way so it lines up.
            rec_name = src.stem
            if rec_name in frame_index_map:
                print(f"    [WARN] duplicate recording name '{rec_name}'; "
                      f"video_frame_indexes entry overwritten.")
            frame_index_map[rec_name] = result["frames"]
            if result["interpolated"]:
                interpolated_recordings.append(rec_name)

    # Write the video_frame_indexes mapping next to the downsampled data.
    if frame_index_map:
        pkl_path = out_root / "video_frame_indexes.pkl"
        json_path = out_root / "video_frame_indexes.json"
        with pkl_path.open("wb") as f:
            pickle.dump(frame_index_map, f)
        with json_path.open("w") as f:
            json.dump(frame_index_map, f)
        print(f"\nWrote video_frame_indexes for {len(frame_index_map)} "
              f"recording(s):\n  {pkl_path}\n  {json_path}")
        print(
            "\nIn your KP-MoSeq script, load the downsampled keypoints, then:\n"
            "    import pickle, numpy as np\n"
            f"    with open(r'{pkl_path}', 'rb') as f:\n"
            "        vfi = {k: np.array(v) for k, v in pickle.load(f).items()}\n"
            "    kpms.update_config(project_dir, fps=10)\n"
            "    kpms.noise_calibration(project_dir, coordinates, confidences,\n"
            "                           video_frame_indexes=vfi, **config())\n"
            "    # and pass video_frame_indexes=vfi to generate_grid_movies too."
        )

    if interpolated_recordings:
        print(
            f"\nNOTE: {len(interpolated_recordings)} recording(s) were "
            f"INTERPOLATED to {TARGET_FPS} FPS (non-integer source ratio). "
            "Their video_frame_indexes are the NEAREST real frame (sub-frame "
            "error), fine for grid movies but slightly approximate for "
            "calibration. For the cleanest calibration, learn slope/intercept "
            "from the exactly-decimated recordings only, e.g.:\n"
            "    interp = set(%r)\n"
            "    cal_coords = {k: v for k, v in coordinates.items() if k not in interp}\n"
            "    cal_confs  = {k: v for k, v in confidences.items() if k not in interp}\n"
            "    cal_vfi    = {k: v for k, v in vfi.items() if k not in interp}\n"
            "    kpms.noise_calibration(project_dir, cal_coords, cal_confs,\n"
            "                           video_frame_indexes=cal_vfi, **config())\n"
            "  (the learned slope/intercept apply to all recordings.)"
            % interpolated_recordings
        )

    # Per-session metadata (one row per recording) + FPS distribution.
    if meta_records:
        from collections import Counter
        meta_path = out_root / "downsample_metadata.csv"
        cols = ["dataset", "recording", "source_fps", "target_fps", "method",
                "orig_frames", "kept_frames", "fps_source", "relpath"]
        with meta_path.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            w.writerows(meta_records)

        fps_counts = Counter(round(r["source_fps"], 3) for r in meta_records)
        print(f"\nWrote per-session metadata: {meta_path}")
        print("Videos per source FPS:")
        for fps_val in sorted(fps_counts):
            n = fps_counts[fps_val]
            print(f"  {fps_val:>6g} FPS | {'#' * n} {n}")

    print(f"\nDone. {total_files} file(s) written under: {out_root}")
    if skipped:
        print(f"{skipped} file(s) skipped (see [SKIP] lines above).")
    if total_files == 0:
        print("Nothing was processed -- check the folder paths in INPUTS.")
        sys.exit(1)


if __name__ == "__main__":
    main()


=== C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\unifiedDataset_topview_DLC_singleAnimal\analyzedVideos_trainingSessions_DLC_singleAnimal -> 10 FPS ===
  mouse1010819\topView_DLCtracking_pcutoff_0.8_skeleton\mouse1010819_Day02_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk.h5: 50 FPS (from mouse1010819_Day02_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_meta.pickle), decimate every 5 -> 12737/63684 rows
  mouse1010819\topView_DLCtracking_pcutoff_0.8_skeleton\mouse1010819_Day04_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk.h5: 49.98 FPS (from mouse1010819_Day04_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_meta.pickle), interpolate 49.98->10 FPS (non-integer ratio) -> 12234/61142 rows
  mouse1010819\topView_DLCtracking_pcutoff_0.8_skeleton\mou